# 03 Velocity Sections

Behavioral-median P-wave velocity along the seven profiles, with the
mobile-regolith base and the fresh-bedrock interface and their
one-standard-deviation envelopes across the retained ensemble.

Run `01_calibration_and_validation.ipynb` first so that
`behavioral_line_sections.npz` exists, or use the `outputs/dc_sceua_pc055_lb`
directory supplied with this repository. Without that file the notebook falls
back to the best-fit sections and the uncertainty bands collapse to zero width.

Output: `candidate_A6_compact_vp_turbo_smooth`, written to `_figures_local/`.


In [ ]:
from __future__ import annotations

from pathlib import Path
import os
import sys

import matplotlib as mpl
try:
    get_ipython().run_line_magic("matplotlib", "inline")
except NameError:
    pass

import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
from matplotlib.cm import ScalarMappable
from matplotlib.lines import Line2D
from matplotlib.patches import Patch, Rectangle
import matplotlib.patheffects as pe
import numpy as np
import pandas as pd
from scipy.interpolate import RegularGridInterpolator
from scipy.ndimage import gaussian_filter, gaussian_filter1d

ROOT = Path.cwd().resolve()
if not (ROOT / "src").exists():
    ROOT = Path.cwd().resolve().parents[0]
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.data_io import read_config
from src.dem_tools import sample_dem_along_line
from src.sceua_landlab_rd_rpv import (
    SCEUA_PARAMETER_NAMES,
    build_line_section_from_model,
    build_model_from_candidate,
    get_line_split,
    load_sceua_base_inputs,
    load_sceua_config,
    normalize_sceua_parameter_frame,
)

PROJECT_CONFIG = read_config(ROOT / "config.yaml")
RESULT_DIR = ROOT / PROJECT_CONFIG["sceua_landlab_rd_rpv"]["output_dir"]
DATA_DIR = RESULT_DIR / "data"
FIG_DIR = ROOT / PROJECT_CONFIG["paper_inputs"]["figure_output_dir"]
SAVE_FIGURES = True
FIG_DPI = 600

TRAINING_LINES = [
    "TL1",
    "TL2",
    "TL3",
    "TL4",
    "TL5",
]
VALIDATION_LINES = ["TL6", "TL7"]
ALL_LINES = TRAINING_LINES + VALIDATION_LINES
LINE_LABELS = {
    "TL1": "TL-1",
    "TL2": "TL-2",
    "TL3": "TL-3",
    "TL4": "TL-4",
    "TL5": "TL-5",
    "TL6": "TL-6",
    "TL7": "TL-7",
}
DATASET_COLORS = {"calibration": "#1f78b4", "validation": "#d95f02"}

DEFAULT_VP_CMAP = "turbo"
VP_CMAP = DEFAULT_VP_CMAP
# Best-fit mobile regolith is mostly 346-620 m/s. Use one smooth power-law
# normalization instead of a hard piecewise break so the 350-600 m/s interval
# changes gradually while still retaining shallow low-Vp contrast.
VP_DISPLAY_MIN_M_PER_S = 350.0
VP_DISPLAY_MAX_M_PER_S = 4000.0
VP_POWER_GAMMA = 0.62
VP_COLORBAR_TICKS = [350, 550, 1000, 1550, 2250, 3100, 4000]
VP_NORM = mcolors.PowerNorm(
    gamma=VP_POWER_GAMMA,
    vmin=VP_DISPLAY_MIN_M_PER_S,
    vmax=VP_DISPLAY_MAX_M_PER_S,
)
FRESH_BEDROCK_FILL_VP_M_PER_S = 3600.0
FRESH_BEDROCK_VP_BUFFER_M = 6.0
UNCERTAINTY_FILL_COLOR = "#d9d9d9"
UNCERTAINTY_EDGE_COLOR = "0.25"
REGOLITH_UNCERTAINTY_COLOR = "#2ca25f"
DISPLAY_SMOOTH_SIGMA_POINTS = 2.0
VP_DISPLAY_SMOOTH_SIGMA_DEPTH = 0.45
VP_DISPLAY_SMOOTH_SIGMA_DISTANCE = 1.15
SECTION_DEPTH_MAX_M = 30.0
SECTION_MIN_DISPLAY_DEPTH_M = 8.0
SECTION_FRESHBEDROCK_MARGIN_M = 2.0
SECTION_FRESH_INTERFACE_PERCENTILE = 88.0
LINE_DISPLAY_DEPTH_OVERRIDE_M = {}
SECTION_TOP_PADDING_M = 1.5


def windows_safe_path(path: str | Path) -> str:
    """Return a path string that can read or write long paths on Windows."""
    path = Path(path)
    if os.name != "nt":
        return str(path)
    text = str(path.resolve())
    if len(text) < 240 or text.startswith("\\\\?\\"):
        return text
    if text.startswith("\\\\"):
        return "\\\\?\\UNC\\" + text[2:]
    return "\\\\?\\" + text


def output_exists(path: str | Path) -> bool:
    return os.path.exists(windows_safe_path(path))


def read_output_csv(path: str | Path, **kwargs) -> pd.DataFrame:
    return pd.read_csv(windows_safe_path(path), **kwargs)


def load_output_npz(path: str | Path) -> dict[str, np.ndarray]:
    with np.load(windows_safe_path(path), allow_pickle=True) as npz:
        return {key: npz[key].copy() for key in npz.files}


def save_candidate(fig: mpl.figure.Figure, name: str) -> None:
    """Optionally save a candidate figure while keeping inline display as default."""
    if not SAVE_FIGURES:
        return
    FIG_DIR.mkdir(parents=True, exist_ok=True)
    fig.savefig(windows_safe_path(FIG_DIR / f"{name}.png"), dpi=FIG_DPI, bbox_inches="tight")
    fig.savefig(windows_safe_path(FIG_DIR / f"{name}.pdf"), bbox_inches="tight")


def strip_axes(ax) -> None:
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(False)

mpl.rcParams.update({
    "figure.dpi": 130,
    "savefig.dpi": FIG_DPI,
    "font.family": "Arial",
    "font.size": 8,
    "axes.titlesize": 8,
    "axes.labelsize": 8,
    "xtick.labelsize": 7,
    "ytick.labelsize": 7,
    "legend.fontsize": 7,
    "axes.linewidth": 0.6,
    "xtick.major.width": 0.6,
    "ytick.major.width": 0.6,
    "xtick.major.size": 2.5,
    "ytick.major.size": 2.5,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

required = [
    DATA_DIR / "best_fit_parameters.csv",
    DATA_DIR / "behavioral_parameter_sets.csv",
    DATA_DIR / "behavioral_3d_uncertainty.npz",
    DATA_DIR / "paper_line_fit_summary.csv",
]
missing = [str(path) for path in required if not output_exists(path)]
if missing:
    raise FileNotFoundError("Missing required notebook 11 outputs:\n" + "\n".join(missing))

BEHAVIORAL_LINE_SECTIONS_PATH = DATA_DIR / "behavioral_line_sections.npz"
HAS_BEHAVIORAL_LINE_SECTIONS = output_exists(BEHAVIORAL_LINE_SECTIONS_PATH)

print(f"Reading results from: {RESULT_DIR.relative_to(ROOT)}")
print(f"SAVE_FIGURES={SAVE_FIGURES}")
print(f"behavioral_line_sections available: {HAS_BEHAVIORAL_LINE_SECTIONS}")

## Seismic Line Roles

TL-1 to TL-5 are the calibration lines that enter the SCE-UA objective. TL-6 and
TL-7 are held out; they are shown with the same calibrated model and behavioral
interface uncertainty.



In [ ]:
best_parameters = normalize_sceua_parameter_frame(read_output_csv(DATA_DIR / "best_fit_parameters.csv")).iloc[0]
behavioral_sets = normalize_sceua_parameter_frame(read_output_csv(DATA_DIR / "behavioral_parameter_sets.csv"))
line_fit = read_output_csv(DATA_DIR / "paper_line_fit_summary.csv")
uncertainty_3d = load_output_npz(DATA_DIR / "behavioral_3d_uncertainty.npz")
behavioral_line_sections = (
    load_output_npz(BEHAVIORAL_LINE_SECTIONS_PATH)
    if HAS_BEHAVIORAL_LINE_SECTIONS
    else None
)
best_candidate = {name: float(best_parameters[name]) for name in SCEUA_PARAMETER_NAMES}

config = read_config(ROOT / "config.yaml")
sceua_config = load_sceua_config(config)
training_lines, validation_lines = get_line_split(config)
assert training_lines == TRAINING_LINES
assert validation_lines == VALIDATION_LINES
assert sceua_config["output_dir"] == str(RESULT_DIR.relative_to(ROOT)).replace("\\", "/")

base_inputs = load_sceua_base_inputs(config, ROOT, line_names=ALL_LINES)
dem = base_inputs["dem"]
assert base_inputs["rpv_params"]["soil"]["critical_porosity"] == 0.55
assert base_inputs["rpv_params"]["soil"]["hertz_mindlin_bound"] == "lower"
assert base_inputs["rpv_params"]["weathered_bedrock"]["alpha_top"] == 0.015
assert base_inputs["rpv_params"]["fresh_bedrock"]["alpha"] == 0.015
best_model = build_model_from_candidate(best_candidate, base_inputs, config)

print("Best-fit parameters")
display(pd.DataFrame([best_candidate]))
print(f"Behavioral sets available: {len(behavioral_sets)}")
print(f"Best model Vp grid shape: {best_model['vp'].shape}")
print(
    "Best-fit Vp ranges (m/s): "
    f"mobile regolith {best_parameters['min_Vp_m_per_s_soil']:.1f}-"
    f"{best_parameters['max_Vp_m_per_s_soil']:.1f}, "
    f"weathered bedrock {best_parameters['min_Vp_m_per_s_weathered']:.1f}-"
    f"{best_parameters['max_Vp_m_per_s_weathered']:.1f}, "
    f"fresh bedrock {best_parameters['mean_Vp_m_per_s_fresh']:.1f}"
)

In [ ]:
def line_dataset(line_id: str) -> str:
    return "calibration" if line_id in TRAINING_LINES else "validation"


def line_rmse_s(line_id: str) -> float:
    if "line" not in line_fit.columns or "rmse_s" not in line_fit.columns:
        return np.nan
    values = line_fit.loc[line_fit["line"].eq(line_id), "rmse_s"]
    return float(values.iloc[0]) if len(values) else np.nan


def safe_line_key(line_id: str) -> str:
    return "".join(ch if ch.isalnum() or ch == "_" else "_" for ch in line_id)


def dem_surface_for_line(line_id: str) -> np.ndarray:
    line_xy = np.asarray(base_inputs["lines"][line_id]["line_xy"], dtype=float)
    surface = sample_dem_along_line(line_xy[:, 0], line_xy[:, 1], dem)
    if not np.all(np.isfinite(surface)):
        n_bad = int(np.sum(~np.isfinite(surface)))
        raise ValueError(f"DEM sampling failed for {line_id}: {n_bad} non-finite elevation values.")
    return surface


def restore_section_surface_from_dem(line_id: str, section: dict[str, np.ndarray]) -> dict[str, np.ndarray]:
    out = dict(section)
    out["surface_elevation"] = dem_surface_for_line(line_id)
    return out


def extract_sections_for_model(model: dict) -> dict[str, dict[str, np.ndarray]]:
    return {
        line_id: restore_section_surface_from_dem(
            line_id,
            build_line_section_from_model(model, base_inputs["lines"][line_id], base_inputs),
        )
        for line_id in ALL_LINES
    }


def load_behavioral_line_section_outputs(
    arrays: dict[str, np.ndarray],
) -> tuple[dict[str, dict[str, np.ndarray]], dict[str, dict[str, dict[str, np.ndarray]]]]:
    """Load per-line behavioral median Vp and interface uncertainty from notebook 11."""
    sections = {}
    envelopes = {}
    for line_id in ALL_LINES:
        key = safe_line_key(line_id)
        required = [
            f"{key}__distance",
            f"{key}__depth",
            f"{key}__surface_elevation",
            f"{key}__vp_p50",
            f"{key}__phi_p50",
            f"{key}__phi_std",
            f"{key}__H_soil_p50",
            f"{key}__H_soil_std",
            f"{key}__D_fresh_p50",
            f"{key}__D_fresh_std",
        ]
        missing = [name for name in required if name not in arrays]
        if missing:
            raise KeyError(f"behavioral_line_sections.npz is missing {line_id} fields: {missing}")
        sections[line_id] = {
            "distance": np.asarray(arrays[f"{key}__distance"], dtype=float),
            "depth": np.asarray(arrays[f"{key}__depth"], dtype=float),
            "surface_elevation": np.asarray(arrays[f"{key}__surface_elevation"], dtype=float),
            "vp": np.asarray(arrays[f"{key}__vp_p50"], dtype=float),
            "phi": np.asarray(arrays[f"{key}__phi_p50"], dtype=float),
            "H_soil": np.asarray(arrays[f"{key}__H_soil_p50"], dtype=float),
            "H_weathered": np.asarray(arrays.get(f"{key}__H_weathered_p50", np.nan), dtype=float),
            "D_fresh": np.asarray(arrays[f"{key}__D_fresh_p50"], dtype=float),
        }
        envelopes[line_id] = {
            "phi": {
                "center": np.asarray(arrays[f"{key}__phi_p50"], dtype=float),
                "std": np.asarray(arrays[f"{key}__phi_std"], dtype=float),
                "center_label": "behavioral median",
            },
            "H_soil": {
                "center": np.asarray(arrays[f"{key}__H_soil_p50"], dtype=float),
                "std": np.asarray(arrays[f"{key}__H_soil_std"], dtype=float),
                "center_label": "behavioral median",
            },
            "D_fresh": {
                "center": np.asarray(arrays[f"{key}__D_fresh_p50"], dtype=float),
                "std": np.asarray(arrays[f"{key}__D_fresh_std"], dtype=float),
                "center_label": "behavioral median",
            },
        }
    return sections, envelopes


def best_fit_interface_envelopes(
    sections: dict[str, dict[str, np.ndarray]],
) -> dict[str, dict[str, dict[str, np.ndarray]]]:
    """Build zero-width interface envelopes from best-fit line sections."""
    envelopes = {}
    for line_id, section in sections.items():
        h_soil = np.asarray(section["H_soil"], dtype=float)
        d_fresh = np.asarray(section["D_fresh"], dtype=float)
        envelopes[line_id] = {
            "H_soil": {
                "center": h_soil,
                "std": np.zeros_like(h_soil),
                "center_label": "best fit",
            },
            "D_fresh": {
                "center": d_fresh,
                "std": np.zeros_like(d_fresh),
                "center_label": "best fit",
            },
        }
    return envelopes


best_sections = extract_sections_for_model(best_model)
if behavioral_line_sections is not None:
    behavioral_median_sections, interface_envelopes = load_behavioral_line_section_outputs(
        behavioral_line_sections
    )
    median_interface_sections = behavioral_median_sections
    n_behavioral_line_sets = int(np.asarray(behavioral_line_sections["n_behavioral_sets"]).ravel()[0])
    active_section_source = f"behavioral median from {n_behavioral_line_sets} retained sets"
else:
    behavioral_median_sections = None
    median_interface_sections = best_sections
    interface_envelopes = best_fit_interface_envelopes(best_sections)
    n_behavioral_line_sets = 0
    active_section_source = "best-fit model; optional behavioral_line_sections.npz not found"

print(f"Extracted best-fit Vp sections for {len(best_sections)} lines.")
print(f"Active section source: {active_section_source}.")


In [ ]:
STYLE_PRESETS = {
    "viridis_lowvp_enhanced": {
        "label": "Comparison: smooth viridis active Vp",
        "cmap": "viridis",
        "section_source": "median_interfaces",
    },
    "magma_median_interfaces": {
        "label": "Trial: magma active Vp",
        "cmap": "magma",
        "section_source": "median_interfaces",
    },
    "turbo_median_interfaces": {
        "label": "Recommended: smooth turbo active Vp",
        "cmap": "turbo",
        "section_source": "median_interfaces",
    },
    "jet_median_interfaces": {
        "label": "Trial: jet active Vp",
        "cmap": "jet",
        "section_source": "median_interfaces",
    },
    "magma_bestfit_interfaces": {
        "label": "Comparison: best-fit Vp and interfaces",
        "cmap": "magma",
        "section_source": "best_fit",
    },
}


def sections_for_style(style: dict[str, object]) -> dict[str, dict[str, np.ndarray]]:
    if style.get("section_source") == "best_fit":
        return best_sections
    return median_interface_sections


def _smooth_profile(values: np.ndarray, sigma_points: float = DISPLAY_SMOOTH_SIGMA_POINTS) -> np.ndarray:
    """Smooth a 1-D display profile while preserving NaN locations."""
    arr = np.asarray(values, dtype=float)
    if sigma_points <= 0 or arr.size < 3:
        return arr
    finite = np.isfinite(arr)
    if not np.any(finite):
        return arr
    if np.all(finite):
        return gaussian_filter1d(arr, sigma=sigma_points, mode="nearest")
    x = np.arange(arr.size, dtype=float)
    filled = np.interp(x, x[finite], arr[finite])
    out = gaussian_filter1d(filled, sigma=sigma_points, mode="nearest")
    out[~finite] = np.nan
    return out


def section_display_depth(line_id: str, section: dict[str, np.ndarray]) -> float:
    """Choose a line-specific crop depth from the behavioral fresh-interface envelope."""
    if line_id in LINE_DISPLAY_DEPTH_OVERRIDE_M:
        return float(LINE_DISPLAY_DEPTH_OVERRIDE_M[line_id])
    env = interface_envelopes[line_id]["D_fresh"]
    d_display = np.asarray(env["center"], dtype=float) + np.asarray(env["std"], dtype=float)
    target = max(
        SECTION_MIN_DISPLAY_DEPTH_M,
        float(np.nanpercentile(d_display, SECTION_FRESH_INTERFACE_PERCENTILE))
        + FRESH_BEDROCK_VP_BUFFER_M
        + SECTION_FRESHBEDROCK_MARGIN_M,
    )
    return float(min(SECTION_DEPTH_MAX_M, target))


def _smooth_vp_for_display(vp_plot: np.ndarray) -> np.ndarray:
    """Lightly smooth displayed Vp while preserving NaN masks and model values."""
    arr = np.asarray(vp_plot, dtype=float)
    sigma = (VP_DISPLAY_SMOOTH_SIGMA_DEPTH, VP_DISPLAY_SMOOTH_SIGMA_DISTANCE)
    if max(sigma) <= 0.0 or arr.size == 0:
        return arr
    finite = np.isfinite(arr)
    if not np.any(finite):
        return arr
    weighted = gaussian_filter(np.where(finite, arr, 0.0), sigma=sigma, mode="nearest")
    weights = gaussian_filter(finite.astype(float), sigma=sigma, mode="nearest")
    out = np.divide(weighted, weights, out=np.full_like(arr, np.nan), where=weights > 1.0e-8)
    out[~finite] = np.nan
    return out


def _vp_for_display(vp_plot: np.ndarray) -> np.ma.MaskedArray:
    """Smooth and clip Vp only for display, then mask invalid values."""
    smoothed = _smooth_vp_for_display(vp_plot)
    clipped = np.clip(smoothed, VP_DISPLAY_MIN_M_PER_S, VP_DISPLAY_MAX_M_PER_S)
    return np.ma.masked_invalid(clipped)


def _draw_depth_uncertainty(
    ax,
    distance: np.ndarray,
    surface: np.ndarray,
    center_depth: np.ndarray,
    std_depth: np.ndarray,
    *,
    fill_color: str,
    edge_color: str,
    fill_alpha: float,
    edge_lw: float,
    label: str,
    zorder: int,
) -> tuple[np.ndarray, np.ndarray]:
    shallow = np.maximum(center_depth - std_depth, 0.0)
    deep = center_depth + std_depth
    upper = surface - shallow
    lower = surface - deep
    ax.fill_between(
        distance,
        lower,
        upper,
        color=fill_color,
        alpha=fill_alpha,
        lw=0,
        label=label,
        zorder=zorder,
    )
    ax.plot(
        distance,
        upper,
        color=edge_color,
        lw=edge_lw,
        ls=(0, (2.2, 1.4)),
        alpha=0.9,
        zorder=zorder + 1,
    )
    ax.plot(
        distance,
        lower,
        color=edge_color,
        lw=edge_lw,
        ls=(0, (2.2, 1.4)),
        alpha=0.9,
        zorder=zorder + 1,
    )
    return lower, upper


def plot_vp_section(
    ax,
    line_id: str,
    section: dict[str, np.ndarray],
    *,
    style: dict[str, object] | None = None,
    show_ylabel: bool = True,
    show_xlabel: bool = True,
) -> mpl.collections.QuadMesh:
    style = STYLE_PRESETS["turbo_median_interfaces"] if style is None else style
    distance = np.asarray(section["distance"], dtype=float)
    depth = np.asarray(section["depth"], dtype=float)
    vp = np.asarray(section["vp"], dtype=float)
    display_depth = section_display_depth(line_id, section)
    keep_depth = depth <= display_depth
    depth_plot = depth[keep_depth]
    vp_plot = vp[keep_depth]
    surface = np.asarray(section["surface_elevation"], dtype=float)
    h_soil = np.asarray(section["H_soil"], dtype=float)
    d_fresh = np.asarray(section["D_fresh"], dtype=float)

    h_soil_plot = _smooth_profile(h_soil)
    d_fresh_plot = _smooth_profile(d_fresh)
    fresh_buffer_m = float(style.get("fresh_velocity_buffer_m", FRESH_BEDROCK_VP_BUFFER_M))
    fresh_crop_depth = d_fresh_plot + fresh_buffer_m

    vp_display = _vp_for_display(vp_plot)
    elevation_grid = surface[None, :] - depth_plot[:, None]
    distance_grid = np.tile(distance[None, :], (len(depth_plot), 1))

    zmin = float(np.nanmin(surface - display_depth))
    zmax = float(np.nanmax(surface + SECTION_TOP_PADDING_M))
    im = ax.pcolormesh(
        distance_grid,
        elevation_grid,
        vp_display,
        shading="auto",
        cmap=str(style["cmap"]),
        norm=VP_NORM,
        rasterized=True,
        zorder=1,
    )
    ax.fill_between(
        distance,
        np.full_like(distance, zmin),
        surface - fresh_crop_depth,
        color="white",
        alpha=1.0,
        lw=0,
        zorder=2,
    )

    env = interface_envelopes[line_id]
    h_env = env["H_soil"]
    d_env = env["D_fresh"]
    h_center = _smooth_profile(np.asarray(h_env["center"], dtype=float))
    h_std_display = _smooth_profile(np.asarray(h_env["std"], dtype=float))
    d_center = _smooth_profile(np.asarray(d_env["center"], dtype=float))
    d_std = _smooth_profile(np.asarray(d_env["std"], dtype=float))

    _draw_depth_uncertainty(
        ax,
        distance,
        surface,
        d_center,
        d_std,
        fill_color=UNCERTAINTY_FILL_COLOR,
        edge_color=UNCERTAINTY_EDGE_COLOR,
        fill_alpha=0.44,
        edge_lw=0.65,
        label=r"$D_f \pm 1\sigma$",
        zorder=3,
    )
    _draw_depth_uncertainty(
        ax,
        distance,
        surface,
        h_center,
        h_std_display,
        fill_color=REGOLITH_UNCERTAINTY_COLOR,
        edge_color=REGOLITH_UNCERTAINTY_COLOR,
        fill_alpha=0.42,
        edge_lw=0.75,
        label=r"$H_m \pm 1\sigma$",
        zorder=5,
    )

    ax.plot(distance, surface, color="black", lw=1.0, label="land surface", zorder=8)
    soil_line, = ax.plot(
        distance,
        surface - h_soil_plot,
        color="white",
        lw=1.10,
        alpha=0.98,
        label="mobile-regolith base",
        zorder=9,
    )
    soil_line.set_path_effects([pe.withStroke(linewidth=2.0, foreground="0.35")])
    ax.plot(
        distance,
        surface - d_fresh_plot,
        color="#00bcd4",
        lw=1.30,
        alpha=0.98,
        label="fresh-bedrock interface",
        zorder=9,
    )

    ax.set_ylim(zmin, zmax)
    ax.set_xlim(float(np.nanmin(distance)), float(np.nanmax(distance)))
    if show_ylabel:
        ax.set_ylabel("Elevation (m)")
    else:
        ax.set_ylabel("")
        ax.tick_params(labelleft=False)
    if show_xlabel:
        ax.set_xlabel("Distance along line (m)")
    else:
        ax.set_xlabel("")
        ax.tick_params(labelbottom=False)
    group = line_dataset(line_id)
    ax.set_title(f"{LINE_LABELS[line_id]} ({group}, RMSE={line_rmse_s(line_id):.4f} s)", pad=2)
    strip_axes(ax)
    return im


def add_legend_and_colorbar(fig, panel_ax, style: dict[str, object]) -> None:
    """Use the empty eighth panel for a readable legend and compact colorbar."""
    panel_ax.axis("off")
    mobile_handle = Line2D([0], [0], color="white", lw=1.4, label="mobile-regolith base")
    mobile_handle.set_path_effects([pe.withStroke(linewidth=2.4, foreground="0.35")])
    df_handle = Rectangle(
        (0, 0),
        1,
        1,
        facecolor=mcolors.to_rgba(UNCERTAINTY_FILL_COLOR, 0.44),
        edgecolor=UNCERTAINTY_EDGE_COLOR,
        linewidth=0.8,
        linestyle=(0, (2.2, 1.4)),
        label=r"$D_f \pm 1\sigma$",
    )
    hm_handle = Rectangle(
        (0, 0),
        1,
        1,
        facecolor=mcolors.to_rgba(REGOLITH_UNCERTAINTY_COLOR, 0.42),
        edgecolor=REGOLITH_UNCERTAINTY_COLOR,
        linewidth=0.8,
        linestyle=(0, (2.2, 1.4)),
        label=r"$H_m \pm 1\sigma$",
    )
    legend_handles = [
        mobile_handle,
        Line2D([0], [0], color="#00bcd4", lw=1.4, label="fresh-bedrock interface"),
        df_handle,
        hm_handle,
    ]
    panel_ax.text(
        0.02,
        0.97,
        str(style["label"]),
        transform=panel_ax.transAxes,
        ha="left",
        va="top",
        fontsize=8,
        color="0.15",
        fontweight="bold",
    )
    panel_ax.legend(
        handles=legend_handles,
        loc="upper left",
        bbox_to_anchor=(0.02, 0.82),
        frameon=False,
        fontsize=7,
        handlelength=2.0,
        borderaxespad=0.0,
        labelspacing=0.55,
    )
    panel_ax.text(
        0.02,
        0.34,
        f"Vp colors extend {FRESH_BEDROCK_VP_BUFFER_M:g} m below the fresh-bedrock interface.",
        transform=panel_ax.transAxes,
        ha="left",
        va="top",
        fontsize=7,
        color="0.30",
        wrap=True,
    )
    cax = panel_ax.inset_axes([0.02, 0.080, 0.96, 0.070])
    cbar = fig.colorbar(
        ScalarMappable(norm=VP_NORM, cmap=str(style["cmap"])),
        cax=cax,
        orientation="horizontal",
    )
    tick_values = VP_COLORBAR_TICKS
    cbar.set_ticks(tick_values)
    cbar.set_ticklabels(["" for _ in tick_values])
    cbar.ax.xaxis.set_minor_locator(mpl.ticker.NullLocator())
    cbar.outline.set_linewidth(0.5)
    cbar.ax.tick_params(width=0.5, length=2.0, pad=1.0)
    for tick in tick_values:
        y_offset = -0.48
        cbar.ax.text(
            VP_NORM(tick),
            y_offset,
            f"{tick:g}",
            transform=cbar.ax.transAxes,
            ha="center",
            va="top",
            fontsize=6.3,
            color="0.05",
            clip_on=False,
        )
    cbar.ax.text(
        0.5,
        -1.26,
        "Vp (m s$^{-1}$)",
        transform=cbar.ax.transAxes,
        ha="center",
        va="top",
        fontsize=8,
        color="0.05",
    )


def plot_compact_vp_grid(style_name: str, output_name: str | None = None):
    """Draw the 7-line compact Vp grid using one named style preset."""
    style = STYLE_PRESETS[style_name]
    sections = sections_for_style(style)
    fig, axes = plt.subplots(4, 2, figsize=(7.2, 7.05), constrained_layout=True)
    axes = axes.ravel()
    for i, line_id in enumerate(ALL_LINES):
        ax = axes[i]
        plot_vp_section(
            ax,
            line_id,
            sections[line_id],
            style=style,
            show_ylabel=(i % 2 == 0),
            show_xlabel=(i >= 5),
        )
    add_legend_and_colorbar(fig, axes[-1], style)
    if output_name:
        save_candidate(fig, output_name)
    return fig, axes


## Behavioral-Median Velocity Sections

This version uses the active line-section source selected above. For the new `outputs/dc_sceua_pc055_lb` run, it uses the exported behavioral-median Vp sections when available. The fresh-bedrock velocity slice is shown to `FRESH_BEDROCK_VP_BUFFER_M = 6 m` below the active `D_f` interface, with a smoothed visual crop and light display-only Vp smoothing so the lower edge and raster cells do not appear blocky. The Vp colors use a smooth power-law normalization from 350 to 4000 m/s rather than a hard color break, so the 350-600 m/s mobile-regolith interval changes gradually while still remaining visible.

In [ ]:
fig, axes = plot_compact_vp_grid(
    "turbo_median_interfaces",
    "candidate_A6_compact_vp_turbo_smooth",
)
plt.show()